In [17]:
# 0 Setup: imports, constants, label-free claims
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

from expense_agent.data import load_claims, load_labels
from expense_agent.models import to_claims
from expense_agent.paths import GENERATED_DIR, POLICY_PATH
from expense_agent.policy import load_policy
from expense_agent.rules import run_rules


N_ESTIMATORS = 200 # more trees: steadier scores, slower fit
SEED = 42
DATE_COLUMNS =["expense_date", "submitted_date"]
MIN_GROUP_SIZE = 3 # fewer claims: no reliable personal baseline
REVIEW_SHARE = 0.10 # share of claims reviewer can handle
claims_df = load_claims(GENERATED_DIR)
claims_df[DATE_COLUMNS] = claims_df[DATE_COLUMNS].apply(pd.to_datetime)


assert len(claims_df) == 330
assert claims_df["claim_id"].is_unique
assert "violation_type" not in claims_df.columns, "labels leaked"
print(claims_df[DATE_COLUMNS + ["amount", "category"]].dtypes)

expense_date      datetime64[us]
submitted_date    datetime64[us]
amount                   float64
category                     str
dtype: object


In [18]:
# 1a Date features: submission lag and weekend spending
features = pd.DataFrame(index=claims_df.index)

features["days_to_submit"] = (
    claims_df["submitted_date"] - claims_df["expense_date"]
).dt.days
features["is_weekend"] = (
    claims_df["expense_date"].dt.day_of_week >= 5
).astype(int)

assert features["days_to_submit"].ge(0).all(), "submitted before expense"
assert set(features["is_weekend"].unique()) <= {0, 1}
print(features.describe().round(2))

       days_to_submit  is_weekend
count          330.00      330.00
mean            18.48        0.14
std             16.31        0.35
min              2.00        0.00
25%             11.00        0.00
50%             16.00        0.00
75%             20.00        0.00
max            120.00        1.00


In [19]:
# 1b Amount features: how unusual is this amount for its category?
amount = claims_df["amount"]
by_category = amount.groupby(claims_df["category"])
cat_std = by_category.transform("std").replace(0, np.nan)

features["log_amount"] = np.log1p(amount)
features["amount_vs_cat_median"] = amount / by_category.transform("median")
features["amount_z_in_cat"] = (
    (amount - by_category.transform("mean")) / cat_std
).fillna(0.0)

assert np.isfinite(features.to_numpy()).all(), "NaN or inf in features"
median_ratio = features.groupby(claims_df["category"])[
    "amount_vs_cat_median"
].median()
assert np.allclose(median_ratio, 1.0), median_ratio
print(features.describe().round(2))

       days_to_submit  is_weekend  log_amount  amount_vs_cat_median  \
count          330.00      330.00      330.00                330.00   
mean            18.48        0.14        5.24                  1.22   
std             16.31        0.35        1.14                  1.18   
min              2.00        0.00        2.57                  0.05   
25%             11.00        0.00        4.27                  0.69   
50%             16.00        0.00        5.40                  1.00   
75%             20.00        0.00        6.00                  1.38   
max            120.00        1.00        8.41                 10.64   

       amount_z_in_cat  
count           330.00  
mean             -0.00  
std               0.99  
min              -2.05  
25%              -0.55  
50%              -0.23  
75%               0.39  
max               4.32  


In [20]:
# 1c Employee features: is this claim unusual for this person?
employee = claims_df["employee_id"]
expense_month = claims_df["expense_date"].dt.to_period("M")

features["claims_in_month"] = (
    claims_df["claim_id"].groupby([employee, expense_month]).transform("count")
)

by_emp_cat = amount.groupby([employee, claims_df["category"]])
emp_median = by_emp_cat.transform("median")
emp_count = by_emp_cat.transform("count")
features["amount_vs_emp_median"] = (amount / emp_median).where(
    emp_count >= MIN_GROUP_SIZE, 1.0
)

assert np.isfinite(features.to_numpy()).all(), "NaN or inf in features"
assert features["claims_in_month"].ge(1).all()
fallback_share = (emp_count < MIN_GROUP_SIZE).mean()
print(f"personal baseline missing for {fallback_share:.0%} of claims")
print(features[["claims_in_month", "amount_vs_emp_median"]].describe().round(2))



personal baseline missing for 12% of claims
       claims_in_month  amount_vs_emp_median
count           330.00                330.00
mean              8.45                  1.18
std               3.85                  1.05
min               1.00                  0.07
25%               6.00                  0.72
50%               8.00                  1.00
75%              10.00                  1.25
max              17.00                 10.60


In [21]:
# 1d Feature matrix: lock the column list, check redundancy
FEATURE_COLUMNS = [
    "days_to_submit",
    "is_weekend",
    "log_amount",
    "amount_vs_cat_median",
    "amount_z_in_cat",
    "claims_in_month",
    "amount_vs_emp_median",
]
X = features[FEATURE_COLUMNS]


assert X.shape == (330, len(FEATURE_COLUMNS))
assert np.isfinite(X.to_numpy()).all()
print(X.corr().round(2))

                      days_to_submit  is_weekend  log_amount  \
days_to_submit                  1.00        0.19       -0.08   
is_weekend                      0.19        1.00       -0.07   
log_amount                     -0.08       -0.07        1.00   
amount_vs_cat_median           -0.04       -0.02        0.49   
amount_z_in_cat                -0.01        0.05        0.51   
claims_in_month                -0.25       -0.24        0.05   
amount_vs_emp_median           -0.03       -0.01        0.44   

                      amount_vs_cat_median  amount_z_in_cat  claims_in_month  \
days_to_submit                       -0.04            -0.01            -0.25   
is_weekend                           -0.02             0.05            -0.24   
log_amount                            0.49             0.51             0.05   
amount_vs_cat_median                  1.00             0.83             0.05   
amount_z_in_cat                       0.83             1.00             0.03   
claims_

In [22]:
# 2 IsolationForest: learn "normal" from the features alone
iso_forest = IsolationForest(
    n_estimators=N_ESTIMATORS,
    contamination="auto",
    random_state=SEED,
)
iso_forest.fit(X)

# score_samples: higher = more normal; negate so higher = more anomalous
anomaly_score = pd.Series(
    -iso_forest.score_samples(X), index=X.index, name="anomaly_score"
)

# Same seed, same data -> same scores (reproducibility check)
rerun = IsolationForest(n_estimators=N_ESTIMATORS, random_state=SEED).fit(X)
assert np.allclose(-rerun.score_samples(X), anomaly_score)
assert iso_forest.n_features_in_ == len(FEATURE_COLUMNS)
print(anomaly_score.describe().round(3))

count    330.000
mean       0.430
std        0.071
min        0.348
25%        0.379
50%        0.405
75%        0.462
max        0.737
Name: anomaly_score, dtype: float64


In [23]:
# 3 Threshold by review capacity; inspect the top 10 (no labels)
threshold = anomaly_score.quantile(1 - REVIEW_SHARE)
is_anomaly = anomaly_score >= threshold

scored_claims = (
    claims_df[["claim_id", "employee_id", "category", "amount"]]
    .join(features[FEATURE_COLUMNS])
    .assign(anomaly_score=anomaly_score, is_anomaly=is_anomaly)
)

expected_flags = round(len(scored_claims) * REVIEW_SHARE)
assert abs(is_anomaly.sum() - expected_flags) <= 1, is_anomaly.sum()
print(f"threshold {threshold:.3f}: {is_anomaly.sum()} claims flagged")

show = [
    "claim_id",
    "category",
    "amount",
    "anomaly_score",
    "amount_vs_cat_median",
    "amount_vs_emp_median",
    "days_to_submit",
    "is_weekend",
    "claims_in_month",
]
top10 = scored_claims.nlargest(10, "anomaly_score")
print(top10[show].round(2).to_string(index=False))

threshold 0.537: 33 claims flagged
    claim_id         category  amount  anomaly_score  amount_vs_cat_median  amount_vs_emp_median  days_to_submit  is_weekend  claims_in_month
R-E009-03-L3          Airfare 4491.99           0.74                 10.64                 10.60              13           0                9
R-E001-05-L2      Home Office 1196.21           0.72                  5.43                  5.40              36           1                5
R-E006-08-L1          Airfare 4028.90           0.69                  9.54                  7.80              15           0               10
R-E006-04-L2          Airfare 3201.06           0.67                  7.58                  6.19              22           0               14
R-E003-07-L4          Airfare 3281.92           0.66                  7.77                  6.79              22           0                8
R-E004-05-L4            Meals  179.41           0.62                  4.07                  5.16              22 

In [24]:
# 4a Evaluate against the answer key (labels enter here, never earlier)
LABEL_COLUMNS = ["claim_id", "in_policy", "violation_type"]


evaluated = scored_claims.merge(
    load_labels(GENERATED_DIR)[LABEL_COLUMNS],
    on="claim_id",
    validate="one_to_one",
)
out_of_policy = ~evaluated["in_policy"]
flagged = evaluated["is_anomaly"]

base_rate = out_of_policy.mean()
precision = out_of_policy[flagged].mean()
recall = flagged[out_of_policy].mean()
max_recall = flagged.sum() / out_of_policy.sum()

assert len(evaluated) == len(scored_claims) == 330
print(f"base rate (random flagging): {base_rate:.1%}")
print(f"precision: {precision:.1%} (lift {precision / base_rate:.1f}x)")
print(f"recall: {recall:.1%} (ceiling at this capacity: {max_recall:.1%})")


base rate (random flagging): 24.8%
precision: 72.7% (lift 2.9x)
recall: 29.3% (ceiling at this capacity: 40.2%)


In [25]:
# 4b-1 Recall by violation type: what can a row-level model see?
by_type = (
    evaluated[out_of_policy]
    .groupby("violation_type")
    .agg(
        claims=("claim_id", "size"),
        flagged=("is_anomaly", "sum"),
        median_score=("anomaly_score", "median"),
    )
    .assign(recall=lambda df: df["flagged"] / df["claims"])
    .sort_values("recall", ascending=False)
)

assert by_type["claims"].sum() == out_of_policy.sum() == 82
assert by_type["flagged"].sum() == 24
print(by_type.round(2).to_string())

                     claims  flagged  median_score  recall
violation_type                                            
premium_class             6        6          0.66    1.00
late_submission          11       10          0.58    0.91
over_cap                 16        6          0.50    0.38
duplicate                 7        1          0.39    0.14
prohibited_category      19        1          0.40    0.05
missing_receipt          14        0          0.39    0.00
personal_extension        6        0          0.49    0.00
split_transaction         3        0          0.46    0.00


In [26]:
# 4b-2 Overlap with the rules: what does the model add?
rule_results = run_rules(
    to_claims(load_claims(GENERATED_DIR)), load_policy(POLICY_PATH)
)
compared = evaluated.merge(
    rule_results[["claim_id", "rule_decision"]],
    on="claim_id",
    validate="one_to_one",
)
positives = compared[~compared["in_policy"]]
rules_deny = positives["rule_decision"] == "deny"

# Create a crosstab to analyze the overlap between rule denials and 
# model-flagged anomalies
overlap = pd.crosstab(
    np.where(rules_deny, "rules deny", "rules miss/review"),
    np.where(positives["is_anomaly"], "flagged", "not flagged"),
    rownames=["rules"],
    colnames=["model"],
    margins=True,
)

# Ensure the number of positive cases matches the expected count
assert len(positives) == 82
assert (~rules_deny).sum() == 23, "rules count drifted from notebook 03"
print(overlap, end="\n\n")

new_catches = positives[~rules_deny & positives["is_anomaly"]]
print(
    new_catches[
        ["claim_id", "category", "amount", "violation_type",
         "rule_decision", "anomaly_score"]
    ].round(3).to_string(index=False)
)

model              flagged  not flagged  All
rules                                       
rules deny              23           36   59
rules miss/review        1           22   23
All                     24           58   82

    claim_id category  amount      violation_type rule_decision  anomaly_score
R-E001-08-L4  Lodging    21.3 prohibited_category        review          0.543


In [27]:
# 4b-3 The cost side: what do the rules say about the 9 false alarms?
false_alarms = compared[compared["in_policy"] & compared["is_anomaly"]]

assert len(false_alarms) == 9
assert not false_alarms["rule_decision"].eq("deny").any(), "rules FP?"
print(false_alarms["rule_decision"].value_counts().to_string(), end="\n\n")

alarm_columns = [
    "claim_id",
    "category",
    "amount",
    "rule_decision",
    "anomaly_score",
    "amount_vs_cat_median",
    "days_to_submit",
]
print(
    false_alarms[alarm_columns]
    .sort_values("anomaly_score", ascending=False)
    .round(2)
    .to_string(index=False)
)

rule_decision
approve    8
review     1

    claim_id         category  amount rule_decision  anomaly_score  amount_vs_cat_median  days_to_submit
R-E004-02-L4 Ground Transport  355.26       approve           0.60                  5.59              22
R-E006-01-L2 Ground Transport  376.79       approve           0.59                  5.93              11
R-E003-06-L1         Software   21.09       approve           0.57                  0.13              21
R-E005-09-L4 Ground Transport  379.33       approve           0.57                  5.97               7
R-E002-05-L3         Software   58.23       approve           0.56                  0.35              37
R-E008-02-L2         Software   29.11       approve           0.55                  0.18              28
R-E002-02-L2         Software   12.09       approve           0.55                  0.07              33
R-E003-09-L4         Software  266.98       approve           0.55                  1.61              32
R-E010-09-L2  

In [31]:
# 4c Compare feature sets: refit and score each one like 4a-4b-3

AMOUNT_FEATURES = [
    "log_amount",
    "amount_vs_cat_median",
    "amount_z_in_cat",
    "amount_vs_emp_median",
]
# 33 claims: the fixed review budget
REVIEW_COUNT = round(len(X) * REVIEW_SHARE)

# Rows must line up: compared (labels + rules) vs X (features)
assert (compared["claim_id"]
        .to_numpy() == claims_df["claim_id"].to_numpy()).all()
is_bad = (~compared["in_policy"]).to_numpy()
rules_miss = (compared["rule_decision"] != "deny").to_numpy()

def score_feature_set(columns: list[str], seed: int = SEED
) -> dict[str, float]:
    """Refit IsolationForest on `columns`; score its top-10% flags vs labels."""
    model = IsolationForest(
        n_estimators= N_ESTIMATORS,
        contamination="auto",
        random_state=seed,
    )
    scores = -model.fit(X[columns]).score_samples(X[columns])

    # Exactly REVIEW_COUNT flags; ties at the cutoff go to the earlier row
    top = np.argsort(-scores, kind="stable")[:REVIEW_COUNT]
    flags = np.zeros(len(scores), dtype=bool)
    flags[top] = True

    hits = flags & is_bad
    return {
        "n_features": len(columns),
        "flagged": int(flags.sum()),
        "true_flags": int(hits.sum()),
        "precision": hits.sum() / flags.sum(),
        "recall": hits.sum(0 ) / is_bad.sum(),
        "new_catches": int((hits & rules_miss).sum()),
        "false_alarms": int((flags & ~is_bad).sum()),
    }


feature_sets = {"baseline (all 7)": FEATURE_COLUMNS}
for column in AMOUNT_FEATURES:
    feature_sets[f"drop {column}"] = [
        c for c in FEATURE_COLUMNS if c != column
    ]
feature_sets["amount only"] = AMOUNT_FEATURES
feature_sets["no amount"] = [
    c for c in FEATURE_COLUMNS if c not in AMOUNT_FEATURES
]

ablation = pd.DataFrame.from_dict(
    {name: score_feature_set(cols) for name, cols in feature_sets.items()},
    orient="index",
)

# The baseline row must reproduce 4a-4b-3 exactly, or the harness is wrong
baseline = ablation.loc["baseline (all 7)"]
assert baseline["true_flags"] == 24
assert baseline["false_alarms"] == 9
assert baseline["new_catches"] == 1
assert np.isclose(baseline["precision"], precision)
print(ablation.round(3).to_string())

                           n_features  flagged  true_flags  precision  recall  new_catches  false_alarms
baseline (all 7)                    7       33          24      0.727   0.293            1             9
drop log_amount                     6       33          25      0.758   0.305            1             8
drop amount_vs_cat_median           6       33          24      0.727   0.293            1             9
drop amount_z_in_cat                6       33          23      0.697   0.280            0            10
drop amount_vs_emp_median           6       33          24      0.727   0.293            1             9
amount only                         4       33          21      0.636   0.256            1            12
no amount                           3       33          16      0.485   0.195            4            17


In [33]:
# 4d Seed check: are the 4c differences real or noise?
SEEDS = range(20)
CANDIDATES = ["baseline (all 7)", "drop log_amount", "no amount"]
METRICS = ["true_flags", "new_catches", "false_alarms"]

seed_runs = pd.DataFrame(
    [
        {"feature_set": name, "seed": seed, **score_feature_set(feature_sets[name], seed)}
        for name in CANDIDATES
        for seed in SEEDS
    ]
)

assert len(seed_runs) == len(CANDIDATES) * len(SEEDS)
assert seed_runs["flagged"].eq(33).all(), "capacity rule drifted"

summary = seed_runs.groupby("feature_set")[METRICS].agg(["mean", "std", "min", "max"])
print(summary.round(1).to_string())

                 true_flags              new_catches              false_alarms             
                       mean  std min max        mean  std min max         mean  std min max
feature_set                                                                                
baseline (all 7)       23.5  0.8  22  25         1.0  0.2   0   1          9.5  0.8   8  11
drop log_amount        23.4  0.5  23  24         0.8  0.4   0   1          9.6  0.5   9  10
no amount              16.2  0.4  16  17         4.0  0.0   4   4         16.8  0.4  16  17


In [35]:
# 4e Rank only rule-approved claims: who catches the false approvals?
BUDGETS = [5, 10, 20, 33]

approved = (compared["rule_decision"] == "approve").to_numpy()
false_approvals = approved & is_bad
approval_bad_rate = false_approvals.sum() / approved.sum()


def approved_ranking(columns: list[str], seed: int = SEED) -> np.ndarray:
    """Return positions of rule-approved claims, most anomalous first."""
    model = IsolationForest(
        n_estimators=N_ESTIMATORS,
        contamination="auto",
        random_state=seed,
    )
    scores = -model.fit(X[columns]).score_samples(X[columns])
    order = np.argsort(-scores, kind="stable")
    return order[approved[order]]


rows = []
for name in CANDIDATES:
    ranking = approved_ranking(feature_sets[name])
    assert len(ranking) == approved.sum()
    for budget in BUDGETS:
        caught = int(false_approvals[ranking[:budget]].sum())
        rows.append({"feature_set": name, "budget": budget, "caught": caught})
for budget in BUDGETS:
    rows.append({"feature_set": "random (expected)", "budget": budget,
                 "caught": budget * approval_bad_rate})

catch_curve = pd.DataFrame(rows).pivot(
    index="feature_set", columns="budget", values="caught"
)
print(f"{approved.sum()} approved claims, {false_approvals.sum()} of them bad "
      f"({approval_bad_rate:.1%})")
print(catch_curve.round(1).to_string())

237 approved claims, 6 of them bad (2.5%)
budget              5    10   20   33
feature_set                          
baseline (all 7)   0.0  0.0  2.0  5.0
drop log_amount    0.0  0.0  1.0  4.0
no amount          2.0  4.0  4.0  5.0
random (expected)  0.1  0.3  0.5  0.8


In [36]:
# 4f The 6 false approvals; what are they, and where does each model rank them?
rank_columns = {}
for name in CANDIDATES:
    ranking = approved_ranking(feature_sets[name])
    rank = np.full(len(X), np.nan)
    rank[ranking] = np.arange(1, len(ranking) + 1) # 1 = most anomalous approved
    rank_columns[f"rank: {name}"] = rank
ranks = pd.DataFrame(rank_columns, index=compared.index)


detail_columns = [
    "claim_id",
    "category",
    "amount",
    "violation_type",
    "days_to_submit",
    "is_weekend",
    "claims_in_month",
]
detail = compared.loc[false_approvals, detail_columns].join(ranks)

assert len(detail) == false_approvals.sum() == 6
print(detail.sort_values("rank: no amount").to_string(index=False))

    claim_id category  amount     violation_type  days_to_submit  is_weekend  claims_in_month  rank: baseline (all 7)  rank: drop log_amount  rank: no amount
R-E008-05-L5  Lodging  398.13 personal_extension              17           1               13                    20.0                   22.0              2.0
R-E004-04-L1  Lodging  441.32 personal_extension               6           1               11                    23.0                   24.0              3.0
R-E009-10-L2  Lodging  495.91 personal_extension               2           1                3                    19.0                   17.0              6.0
R-E003-01-L3  Lodging  398.24 personal_extension               2           1                8                    27.0                   28.0              7.0
R-E004-10-L2  Lodging  353.71 personal_extension              11           1                4                    31.0                   35.0             25.0
R-E008-04-L3  Lodging  306.99 personal_extension    

In [37]:
# 4g Simpler alternative: is "weekend lodging" all the model learned?
weekend_lodging = (
    compared["category"].eq("Lodging") & compared["is_weekend"].eq(1)
).to_numpy()
rule_flags = approved & weekend_lodging
rule_hits = rule_flags & is_bad

print(f"candidate rule flags {rule_flags.sum()} approved claims, "
      f"catches {rule_hits.sum()} of {false_approvals.sum()} false approvals")
print(f"rule precision: {rule_hits.sum() / rule_flags.sum():.0%} "
      f"vs no-amount model at budget 10: {4 / 10:.0%}")

candidate rule flags 6 approved claims, catches 6 of 6 false approvals
rule precision: 100% vs no-amount model at budget 10: 40%
